In [228]:
import json
import pandas as pd
data_path = '../data/Phase_1/train.json'
with open(data_path) as f:
    data = json.load(f)

len(data)

2000

In [229]:
data_path_test = '../data/Phase_1/test.json'
with open(data_path_test) as f:
    data_test = json.load(f)

len(data_test)

28

In [230]:
import io

def parse_csv_block(text):
    if text is None or text.strip() == "":
        return None
    return pd.read_csv(io.StringIO(text), sep="|")

In [231]:
rows = []

for s in data:
    row = {}
    
    # ---------- label ----------
    row["tag"] = s.get("tag")
    row["num_options"] = len(s["task"]["options"])
    row["answer"] = s.get("answer")
    
    # ---------- context ----------
    ctx = s["context"]
    row["context_description"] = ctx["description"]
    
    net = ctx["wireless_network_information"]
    row["network_type"] = net["network_type"]
    row["num_base_stations"] = int(net["num_base_stations"])
    row["mobility_scenario"] = net["mobility_scenario"]
    
    # ---------- data ----------
    d = s["data"]
    
    def get_shape(df):
        if df is None:
            return 0, 0
        return df.shape[1], df.shape[0]  # cols, rows
    
    # user plane
    df = parse_csv_block(d.get("user_plane_data"))
    row["user_plane_cols"], row["user_plane_rows"] = get_shape(df)
    
    # network config
    df = parse_csv_block(d.get("network_configuration_data"))
    row["config_cols"], row["config_rows"] = get_shape(df)
    
    # signaling
    df = parse_csv_block(d.get("signaling_plane_data"))
    row["signaling_cols"], row["signaling_rows"] = get_shape(df)
    
    # traffic
    df = parse_csv_block(d.get("traffic_data"))
    row["traffic_cols"], row["traffic_rows"] = get_shape(df)
    
    # MR
    df = parse_csv_block(d.get("mr_data"))
    row["mr_cols"], row["mr_rows"] = get_shape(df)
    
    # ---------- metadata ----------
    row["notes"] = d.get("notes")
    row["collection_method"] = d.get("collection_method")
    
    # ---------- tools ----------
    t = s["tools"]
    row["tools_description"] = t.get("description")
    row["tools_capabilities"] = ",".join(t.get("capabilities", []))
    
    rows.append(row)

df_meta = pd.DataFrame(rows)
df_meta.head()

,tag,num_options,answer,context_description,network_type,num_base_stations,mobility_scenario,user_plane_cols,user_plane_rows,config_cols,...,signaling_cols,signaling_rows,traffic_cols,traffic_rows,mr_cols,mr_rows,notes,collection_method,tools_description,tools_capabilities
0,multiple-answer,22,C2|C8|C11|C16,A network engineer conducted drive testing wit...,5G,5,vehicle-based drive test,32,12,28,...,3,22,14,5,14,5,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval
1,single-answer,22,C9,A network engineer conducted drive testing wit...,5G,4,vehicle-based drive test,32,12,28,...,3,10,14,4,14,8,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval
2,single-answer,22,C16,A network engineer conducted drive testing wit...,5G,4,vehicle-based drive test,32,12,28,...,3,10,14,4,14,7,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval
3,single-answer,22,C10,A network engineer conducted drive testing wit...,5G,4,vehicle-based drive test,32,12,28,...,3,10,14,4,14,6,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval
4,single-answer,22,C3,A network engineer conducted drive testing wit...,5G,4,vehicle-based drive test,32,12,28,...,3,10,14,4,14,6,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval


In [232]:
df_meta.describe()

,num_options,num_base_stations,user_plane_cols,user_plane_rows,config_cols,config_rows,signaling_cols,signaling_rows,traffic_cols,traffic_rows,mr_cols,mr_rows
count,2000.0,2000.000000,2000.0,2000.0,2000.0,2000.000000,2000.0,2000.000000,2000.0,2000.000000,2000.0,2000.000000
mean,22.0,5.138000,32.0,12.0,28.0,5.138000,3.0,10.369000,14.0,5.138000,14.0,6.439000
std,0.0,2.944026,0.0,0.0,0.0,2.944026,0.0,2.938393,0.0,2.944026,0.0,1.129565
min,22.0,4.000000,32.0,12.0,28.0,4.000000,3.0,4.000000,14.0,4.000000,14.0,5.000000
25%,22.0,4.000000,32.0,12.0,28.0,4.000000,3.0,10.000000,14.0,4.000000,14.0,5.000000
50%,22.0,4.000000,32.0,12.0,28.0,4.000000,3.0,10.000000,14.0,4.000000,14.0,6.000000
75%,22.0,4.000000,32.0,12.0,28.0,4.000000,3.0,12.000000,14.0,4.000000,14.0,7.000000
max,22.0,13.000000,32.0,12.0,28.0,13.000000,3.0,22.000000,14.0,13.000000,14.0,8.000000


In [233]:
df_meta["num_answers"] = df_meta["answer"].apply(lambda x: len(x.split("|")) if x else 0)

df_meta["num_answers"].value_counts()

num_answers
1    1701
2     228
4      71
Name: count, dtype: int64

In [234]:
df_meta["tag"].value_counts()

tag
single-answer      1701
multiple-answer     299
Name: count, dtype: int64

In [235]:
df_meta["num_base_stations"].value_counts()

num_base_stations
4     1684
13     245
5       71
Name: count, dtype: int64

In [236]:
df_meta[[
    "user_plane_rows",
    "signaling_rows",
    "traffic_rows",
    "mr_rows"
]].describe()

,user_plane_rows,signaling_rows,traffic_rows,mr_rows
count,2000.0,2000.000000,2000.000000,2000.000000
mean,12.0,10.369000,5.138000,6.439000
std,0.0,2.938393,2.944026,1.129565
min,12.0,4.000000,4.000000,5.000000
25%,12.0,10.000000,4.000000,5.000000
50%,12.0,10.000000,4.000000,6.000000
75%,12.0,12.000000,4.000000,7.000000
max,12.0,22.000000,13.000000,8.000000


In [237]:
df_meta[[
    "user_plane_cols",
    "signaling_cols",
    "traffic_cols",
    "mr_rows"
]].describe()

,user_plane_cols,signaling_cols,traffic_cols,mr_rows
count,2000.0,2000.0,2000.0,2000.000000
mean,32.0,3.0,14.0,6.439000
std,0.0,0.0,0.0,1.129565
min,32.0,3.0,14.0,5.000000
25%,32.0,3.0,14.0,5.000000
50%,32.0,3.0,14.0,6.000000
75%,32.0,3.0,14.0,7.000000
max,32.0,3.0,14.0,8.000000


In [238]:
df_meta.isnull().mean()

tag                    0.0
num_options            0.0
answer                 0.0
context_description    0.0
network_type           0.0
num_base_stations      0.0
mobility_scenario      0.0
user_plane_cols        0.0
user_plane_rows        0.0
config_cols            0.0
config_rows            0.0
signaling_cols         0.0
signaling_rows         0.0
traffic_cols           0.0
traffic_rows           0.0
mr_cols                0.0
mr_rows                0.0
notes                  0.0
collection_method      0.0
tools_description      0.0
tools_capabilities     0.0
num_answers            0.0
dtype: float64

In [239]:
df_meta['collection_method'].unique()

<ArrowStringArray>
['Drive testing performed by an engineer']
Length: 1, dtype: str

In [240]:
from collections import defaultdict

all_options = []

for s in data:
    for opt in s["task"]["options"]:
        all_options.append({
            "scenario_id": s["scenario_id"],
            "option_id": opt["id"],
            "label": opt["label"]
        })

import pandas as pd
df_options = pd.DataFrame(all_options)

df_options.head()

,scenario_id,option_id,label
0,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C1,Add neighbor relationship between 3267220_2 an...
1,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C2,Decrease transmission power for 3279943_1
2,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C3,Increase transmission power for 3267220_2
3,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C4,Check test server and transmission issues
4,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C5,Decrease CovInterFreqA2RsrpThld and CovInterFr...


In [241]:
df_options

,scenario_id,option_id,label
0,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C1,Add neighbor relationship between 3267220_2 an...
1,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C2,Decrease transmission power for 3279943_1
2,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C3,Increase transmission power for 3267220_2
3,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C4,Check test server and transmission issues
4,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C5,Decrease CovInterFreqA2RsrpThld and CovInterFr...
...,...,...,...
43995,6a686272-80c5-4a77-ba51-192998460176,C18,Increase A3 Offset threshold for 3230521_4
43996,6a686272-80c5-4a77-ba51-192998460176,C19,Check test server and transmission issues
43997,6a686272-80c5-4a77-ba51-192998460176,C20,Insufficient data; more data is needed for jud...
43998,6a686272-80c5-4a77-ba51-192998460176,C21,Decrease CovInterFreqA2RsrpThld and CovInterFr...


In [242]:
import re

def normalize_label(label):
    label = label.lower()
    
    # remove cell IDs like 3279943_1
    label = re.sub(r'\d+_\d+', 'CELL', label)
    
    # remove standalone numbers (angles, thresholds)
    label = re.sub(r'\d+', 'NUM', label)
    
    # clean spaces
    label = re.sub(r'\s+', ' ', label).strip()
    
    return label

df_options["normalized"] = df_options["label"].apply(normalize_label)

In [243]:
df_options

,scenario_id,option_id,label,normalized
0,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C1,Add neighbor relationship between 3267220_2 an...,add neighbor relationship between CELL and CELL
1,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C2,Decrease transmission power for 3279943_1,decrease transmission power for CELL
2,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C3,Increase transmission power for 3267220_2,increase transmission power for CELL
3,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C4,Check test server and transmission issues,check test server and transmission issues
4,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C5,Decrease CovInterFreqA2RsrpThld and CovInterFr...,decrease covinterfreqaNUMrsrpthld and covinter...
...,...,...,...,...
43995,6a686272-80c5-4a77-ba51-192998460176,C18,Increase A3 Offset threshold for 3230521_4,increase aNUM offset threshold for CELL
43996,6a686272-80c5-4a77-ba51-192998460176,C19,Check test server and transmission issues,check test server and transmission issues
43997,6a686272-80c5-4a77-ba51-192998460176,C20,Insufficient data; more data is needed for jud...,insufficient data; more data is needed for jud...
43998,6a686272-80c5-4a77-ba51-192998460176,C21,Decrease CovInterFreqA2RsrpThld and CovInterFr...,decrease covinterfreqaNUMrsrpthld and covinter...


In [244]:
df_options["normalized"].value_counts()

normalized
add neighbor relationship between CELL and CELL                                          4000
decrease transmission power for CELL                                                     4000
increase transmission power for CELL                                                     4000
decrease covinterfreqaNUMrsrpthld and covinterfreqaNUMrsrpthldNUM thresholds for CELL    4000
press down the tilt angle of CELL by NUM degrees                                         4000
modify pdcchoccupiedsymbolnum to NUMsym for CELL                                         4000
adjust the azimuth of CELL by NUM degrees                                                4000
increase aNUM offset threshold for CELL                                                  4000
lift the tilt angle of CELL by NUM degrees                                               4000
decrease aNUM offset threshold for CELL                                                  4000
check test server and transmission issues        

In [245]:
LABEL_SPACE = [
    "add_neighbor",
    "decrease_power",
    "increase_power",
    "decrease_threshold",
    "tilt_down",
    "modify_pdcch",
    "adjust_azimuth",
    "increase_a3",
    "tilt_up",
    "decrease_a3",
    "check_transport",
    "insufficient_data"
]

In [246]:
def map_to_category(norm_label):
    if "neighbor" in norm_label:
        return "add_neighbor"
    if "decrease transmission power" in norm_label:
        return "decrease_power"
    if "increase transmission power" in norm_label:
        return "increase_power"
    if "covinterfreq" in norm_label:
        return "decrease_threshold"
    if "press down the tilt" in norm_label:
        return "tilt_down"
    if "lift the tilt" in norm_label:
        return "tilt_up"
    if "pdcch" in norm_label:
        return "modify_pdcch"
    if "azimuth" in norm_label:
        return "adjust_azimuth"
    if "increase a" in norm_label:
        return "increase_a3"
    if "decrease a" in norm_label:
        return "decrease_a3"
    if "test server" in norm_label:
        return "check_transport"
    if "insufficient" in norm_label:
        return "insufficient_data"
    return "unknown"

In [247]:
import numpy as np

def build_multilabel(scenario):
    id_to_label = {opt["id"]: opt["label"] for opt in scenario["task"]["options"]}
    
    vec = np.zeros(len(LABEL_SPACE))
    
    for cid in scenario["answer"].split("|"):
        label = id_to_label[cid]
        norm = normalize_label(label)
        cat = map_to_category(norm)
        
        if cat in LABEL_SPACE:
            idx = LABEL_SPACE.index(cat)
            vec[idx] = 1
    
    return vec

In [248]:
multi_labels = np.array([build_multilabel(s) for s in data])

multi_labels.shape

(2000, 12)

In [249]:
label_counts = multi_labels.sum(axis=0)

for label, count in zip(LABEL_SPACE, label_counts):
    print(label, count)

add_neighbor 240.0
decrease_power 157.0
increase_power 142.0
decrease_threshold 245.0
tilt_down 157.0
modify_pdcch 240.0
adjust_azimuth 142.0
increase_a3 71.0
tilt_up 254.0
decrease_a3 231.0
check_transport 239.0
insufficient_data 252.0


In [250]:
co_matrix = multi_labels.T @ multi_labels

import pandas as pd
pd.DataFrame(co_matrix, index=LABEL_SPACE, columns=LABEL_SPACE)

,add_neighbor,decrease_power,increase_power,decrease_threshold,tilt_down,modify_pdcch,adjust_azimuth,increase_a3,tilt_up,decrease_a3,check_transport,insufficient_data
add_neighbor,240.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
decrease_power,0.0,157.0,0.0,0.0,157.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0
increase_power,0.0,0.0,142.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0
decrease_threshold,0.0,0.0,0.0,245.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
tilt_down,0.0,157.0,0.0,0.0,157.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0
modify_pdcch,0.0,0.0,0.0,0.0,0.0,240.0,0.0,0.0,0.0,0.0,0.0,0.0
adjust_azimuth,0.0,0.0,142.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0
increase_a3,0.0,71.0,0.0,0.0,71.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0
tilt_up,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,254.0,0.0,0.0,0.0
decrease_a3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,231.0,0.0,0.0


In [251]:
df_options[df_options['normalized']=='increase aNUM offset threshold for CELL']

,scenario_id,option_id,label,normalized
10,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C11,Increase A3 Offset threshold for 3279943_1,increase aNUM offset threshold for CELL
15,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,C16,Increase A3 Offset threshold for 3267220_2,increase aNUM offset threshold for CELL
29,0cd874c0-eab3-4963-a51f-28fc5e004ed8,C8,Increase A3 Offset threshold for 3213751_2,increase aNUM offset threshold for CELL
41,0cd874c0-eab3-4963-a51f-28fc5e004ed8,C20,Increase A3 Offset threshold for 3244912_3,increase aNUM offset threshold for CELL
47,ac53a506-f1dd-404f-bb93-1ffb337ce0e6,C4,Increase A3 Offset threshold for 3211229_4,increase aNUM offset threshold for CELL
...,...,...,...,...
43950,7b3003fb-9bf3-41b7-9eeb-306aa46d8726,C17,Increase A3 Offset threshold for 3236210_3,increase aNUM offset threshold for CELL
43969,bbbf0187-a422-40c0-83b9-93e232cfb17f,C14,Increase A3 Offset threshold for 3264819_3,increase aNUM offset threshold for CELL
43974,bbbf0187-a422-40c0-83b9-93e232cfb17f,C19,Increase A3 Offset threshold for 3238922_2,increase aNUM offset threshold for CELL
43992,6a686272-80c5-4a77-ba51-192998460176,C15,Increase A3 Offset threshold for 3266352_3,increase aNUM offset threshold for CELL


In [252]:
multi_labels

array([[0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(2000, 12))

In [253]:
np.array(df_options['scenario_id'].unique().tolist())[np.where(multi_labels[:, 7] == 1)[0]]

array(['08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e',
       '6fc2c635-2c2a-4f1c-9ce0-db0d4c53b46a',
       '65e93bbb-6649-4c6f-b760-df22ae943159',
       '60725196-a864-4d24-97d2-3ed06582f9a6',
       '4ad88a7d-acb2-4cc0-bd46-650731073814',
       '3da2462e-a0e1-415f-814a-130b8bb10a6b',
       '8d981055-ccdd-43a3-b89c-24b902fcd24b',
       'd6fae17f-d308-4353-a0a9-9173ea702491',
       'a21308d3-6c7d-43ba-907f-e0eec2a8d108',
       'ec76b540-cc3b-45f1-9336-0571d8ee35cb',
       '76da2a11-8643-40d3-8369-928e46cf3f45',
       '63791e7d-0638-4293-8d77-11fa0d01c2c0',
       '0739c14e-bec3-45e6-bc58-8209cc2ceb7f',
       'e05914bb-e6c9-4db4-8294-4d6b6d7f2167',
       '7e6844f5-d1b8-4690-b54c-89bd236b1aed',
       '39c15917-132f-491c-8bda-fc1b5ad1c816',
       '4493a284-8bfc-4e1a-8c64-d5763f3f48dc',
       '2f62bb6a-b3ab-4a0a-823a-ba1d5f618a31',
       '78d4b233-8f06-4dc3-83ce-236548d291c3',
       '416a5366-8e5e-4de0-ad5e-788028ace4f4',
       '5f6743f9-7466-4949-afcc-4abff69b29aa',
       '9a5f6

In [254]:
#check
def build_category_counts(scenario):
    id_to_label = {opt["id"]: opt["label"] for opt in scenario["task"]["options"]}
    
    counts = {k: 0 for k in LABEL_SPACE}
    
    for cid in scenario["answer"].split("|"):
        label = id_to_label[cid]
        norm = normalize_label(label)
        cat = map_to_category(norm)
        
        if cat in counts:
            counts[cat] += 1
    
    return counts
cat_counts = [build_category_counts(s) for s in data]
df_counts = pd.DataFrame(cat_counts)
double_increase_a3 = (df_counts["increase_a3"] >= 2).sum()
double_increase_a3

np.int64(71)

In [255]:
len(np.array(df_options['scenario_id'].unique().tolist())[np.where(multi_labels[:, 7] == 1)[0]])

71

In [256]:
np.where(multi_labels[:, 0] == 1)

(array([  20,   32,   42,   47,   52,   54,   58,   66,   74,   88,   95,
         106,  108,  124,  139,  140,  150,  151,  152,  153,  158,  167,
         178,  190,  191,  203,  205,  210,  223,  225,  248,  251,  259,
         270,  271,  272,  292,  296,  304,  305,  306,  315,  323,  335,
         337,  345,  347,  349,  355,  367,  370,  371,  392,  405,  413,
         417,  424,  426,  427,  436,  444,  454,  468,  492,  493,  499,
         503,  523,  531,  539,  556,  566,  567,  590,  601,  608,  611,
         626,  633,  680,  713,  714,  726,  731,  732,  742,  746,  762,
         769,  773,  781,  782,  787,  790,  797,  798,  808,  832,  878,
         884,  885,  886,  887,  893,  895,  906,  908,  918,  925,  926,
         931,  933,  934,  973,  978,  980,  987,  991,  993, 1004, 1010,
        1036, 1047, 1052, 1055, 1062, 1066, 1069, 1071, 1077, 1078, 1080,
        1084, 1096, 1105, 1113, 1117, 1118, 1122, 1130, 1136, 1143, 1146,
        1156, 1163, 1168, 1199, 1202, 

In [257]:
df_options.iloc[np.where(multi_labels[:, 0] == 1)[0][0]*22]

scenario_id                 30724e42-2186-44df-9fce-eccec04e0470
option_id                                                     C1
label          Insufficient data; more data is needed for jud...
normalized     insufficient data; more data is needed for jud...
Name: 440, dtype: str

In [258]:
df_options.iloc[np.where(multi_labels[:, 3] == 1)[0][0]*22*5]

scenario_id         da7d2a6a-a3ec-43a5-adc9-885ee6844f56
option_id                                             C1
label          Decrease transmission power for 3265550_2
normalized          decrease transmission power for CELL
Name: 1650, dtype: str

In [259]:
df_meta.iloc[np.where(multi_labels[:, 3] == 1)[0]]

,tag,num_options,answer,context_description,network_type,num_base_stations,mobility_scenario,user_plane_cols,user_plane_rows,config_cols,...,signaling_rows,traffic_cols,traffic_rows,mr_cols,mr_rows,notes,collection_method,tools_description,tools_capabilities,num_answers
15,single-answer,22,C14,A network engineer conducted drive testing wit...,5G,13,vehicle-based drive test,32,12,28,...,12,14,13,14,5,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval,1
18,single-answer,22,C12,A network engineer conducted drive testing wit...,5G,13,vehicle-based drive test,32,12,28,...,12,14,13,14,5,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval,1
29,single-answer,22,C4,A network engineer conducted drive testing wit...,5G,13,vehicle-based drive test,32,12,28,...,12,14,13,14,7,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval,1
35,single-answer,22,C17,A network engineer conducted drive testing wit...,5G,13,vehicle-based drive test,32,12,28,...,12,14,13,14,6,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval,1
39,single-answer,22,C22,A network engineer conducted drive testing wit...,5G,13,vehicle-based drive test,32,12,28,...,12,14,13,14,5,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1952,single-answer,22,C6,A network engineer conducted drive testing wit...,5G,13,vehicle-based drive test,32,12,28,...,12,14,13,14,7,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval,1
1956,single-answer,22,C22,A network engineer conducted drive testing wit...,5G,13,vehicle-based drive test,32,12,28,...,12,14,13,14,6,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval,1
1959,single-answer,22,C15,A network engineer conducted drive testing wit...,5G,13,vehicle-based drive test,32,12,28,...,12,14,13,14,5,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval,1
1972,single-answer,22,C12,A network engineer conducted drive testing wit...,5G,13,vehicle-based drive test,32,12,28,...,12,14,13,14,6,,Drive testing performed by an engineer,Analytical tools are provided to retrieve the ...,data retrieval,1


In [260]:
len(multi_labels)

2000

In [261]:
multi_labels.sum(axis=1)

array([3., 1., 1., ..., 1., 1., 1.], shape=(2000,))

In [262]:
multi_labels[0]

array([0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0.])

In [263]:
LABEL_SPACE

['add_neighbor',
 'decrease_power',
 'increase_power',
 'decrease_threshold',
 'tilt_down',
 'modify_pdcch',
 'adjust_azimuth',
 'increase_a3',
 'tilt_up',
 'decrease_a3',
 'check_transport',
 'insufficient_data']

In [264]:
len(multi_labels)

2000

In [265]:
pd.Series(multi_labels.sum(axis=1)).value_counts()

1.0    1701
2.0     228
3.0      71
Name: count, dtype: int64

In [266]:
def has_add_neighbor(scenario):
    id_to_label = {opt["id"]: opt["label"] for opt in scenario["task"]["options"]}
    
    for cid in scenario["answer"].split("|"):
        label = id_to_label[cid]
        norm = normalize_label(label)
        
        if map_to_category(norm) == "add_neighbor":
            return True
    return False


add_neighbor_cases = [s for s in data if has_add_neighbor(s)]

len(add_neighbor_cases)

240

In [267]:
def extract_features(s):
    import io
    
    d = s["data"]
    
    # --- user plane ---
    df_up = parse_csv_block(d["user_plane_data"])
    
    # serving vs neighbor gap
    serving_rsrp = df_up["5G KPI PCell RF Serving SS-RSRP [dBm]"]
    neighbor_rsrp = df_up[
        "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 Filtered Tx BRSRP [dBm]"
    ]
    
    rsrp_gap = (serving_rsrp - neighbor_rsrp).mean()
    
    # throughput
    throughput = df_up["5G KPI PCell Layer2 MAC DL Throughput [Mbps]"]
    
    # PCI switching
    pci_switches = df_up["5G KPI PCell RF Serving PCI"].nunique()
    
    # --- signaling ---
    df_sig = parse_csv_block(d.get("signaling_plane_data"))
    
    if df_sig is not None:
        ho_count = (df_sig["Event Name"] == "NRHandoverAttempt").sum()
        a3_count = (df_sig["Event Name"] == "NREventA3").sum()
    else:
        ho_count = 0
        a3_count = 0
    
    return {
        "rsrp_gap": rsrp_gap,
        "throughput_min": throughput.min(),
        "throughput_mean": throughput.mean(),
        "pci_switches": pci_switches,
        "ho_count": ho_count,
        "a3_count": a3_count
    }

In [268]:
non_add_neighbor_cases = [s for s in data if not has_add_neighbor(s)]

In [269]:
len(non_add_neighbor_cases)

1760

In [270]:
add_feats = pd.DataFrame([extract_features(s) for s in add_neighbor_cases])
non_add_feats = pd.DataFrame([extract_features(s) for s in non_add_neighbor_cases])

In [271]:
add_feats

,rsrp_gap,throughput_min,throughput_mean,pci_switches,ho_count,a3_count
0,-0.037500,26.79,315.913333,1,0,3
1,0.378333,43.73,257.180000,1,0,3
2,0.081667,32.66,315.456667,1,0,3
3,0.158333,28.20,292.205000,1,0,3
4,-0.417500,25.01,314.658333,1,0,3
...,...,...,...,...,...,...
235,-0.503333,31.37,321.041667,1,0,3
236,0.437500,42.45,306.046667,1,0,3
237,0.537500,51.90,322.194167,1,0,3
238,0.245833,35.68,317.848333,1,0,3


In [272]:
non_add_feats

,rsrp_gap,throughput_min,throughput_mean,pci_switches,ho_count,a3_count
0,1.964167,25.61,259.289167,2,3,3
1,4.082500,51.10,265.461667,1,1,1
2,0.585000,25.17,250.120000,2,1,1
3,4.010000,60.28,227.875833,1,1,1
4,5.319167,68.04,280.520833,1,1,1
...,...,...,...,...,...,...
1755,3.864167,48.96,261.834167,1,1,1
1756,4.092500,60.52,248.107500,1,1,1
1757,3.655833,47.15,258.708333,1,1,1
1758,2.875833,56.83,283.061667,1,1,1


In [273]:
for col in add_feats.columns:
    print(f"\n=== {col} ===")
    print("ADD:", add_feats[col].mean())
    print("NON:", non_add_feats[col].mean())


=== rsrp_gap ===
ADD: 0.003243055555555557
NON: 3.285635416666666

=== throughput_min ===
ADD: 36.73241666666666
NON: 49.10842045454546

=== throughput_mean ===
ADD: 317.73371527777783
NON: 247.51748958333337

=== pci_switches ===
ADD: 1.0
NON: 2.006818181818182

=== ho_count ===
ADD: 0.0
NON: 0.9511363636363637

=== a3_count ===
ADD: 3.0
NON: 0.8119318181818181


In [274]:
import io
import numpy as np
import pandas as pd

def extract_features(s):
    d = s["data"]
    feats = {}
    
    # ---------- USER PLANE ----------
    df_up = parse_csv_block(d.get("user_plane_data"))
    
    if df_up is not None and not df_up.empty:
        th = df_up["5G KPI PCell Layer2 MAC DL Throughput [Mbps]"]
        sinr = df_up["5G KPI PCell RF Serving SS-SINR [dB]"]
        rsrp = df_up["5G KPI PCell RF Serving SS-RSRP [dBm]"]
        
        # --- Throughput (enhanced) ---
        feats["throughput_mean"] = th.mean()
        feats["throughput_min"] = th.min()
        feats["throughput_std"] = th.std()
        feats["throughput_p10"] = np.percentile(th, 10)
        feats["throughput_low_ratio"] = (th < 150).mean()
        feats["throughput_drop_ratio"] = th.min() / (th.max() + 1e-6)
        
        # --- RF ---
        feats["sinr_mean"] = sinr.mean()
        feats["sinr_min"] = sinr.min()
        feats["rsrp_mean"] = rsrp.mean()
        
        # --- RF vs Throughput mismatch ---
        feats["rf_good_flag"] = int((feats["rsrp_mean"] > -95) and (feats["sinr_mean"] > 8))
        feats["low_tp_flag"] = int(feats["throughput_mean"] < 150)
        feats["rf_tp_mismatch"] = int(feats["rf_good_flag"] and feats["low_tp_flag"])
        
        # --- PCI switching ---
        feats["pci_switches"] = df_up["5G KPI PCell RF Serving PCI"].nunique()
        
        # --- Neighbor gap ---
        if "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 Filtered Tx BRSRP [dBm]" in df_up.columns:
            neigh = df_up["Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 Filtered Tx BRSRP [dBm]"]
            feats["rsrp_gap"] = (rsrp - neigh).mean()
            feats["serving_better_ratio"] = (rsrp > neigh).mean()
        else:
            feats["rsrp_gap"] = np.nan
            feats["serving_better_ratio"] = np.nan
        
        # --- 🔥 CCE / PDCCH features (CRITICAL) ---
        if "CCE Fail Rate" in df_up.columns:
            cce = df_up["CCE Fail Rate"]
            feats["cce_fail_mean"] = cce.mean()
            feats["cce_fail_max"] = cce.max()
            feats["cce_fail_high_ratio"] = (cce > 0.4).mean()
        else:
            feats["cce_fail_mean"] = np.nan
            feats["cce_fail_max"] = np.nan
            feats["cce_fail_high_ratio"] = np.nan
        
        # --- Scheduling features ---
        if "Grant" in df_up.columns:
            feats["grant_mean"] = df_up["Grant"].mean()
        else:
            feats["grant_mean"] = np.nan
        
        if "5G KPI PCell Layer1 DL RB Num (Including 0)" in df_up.columns:
            feats["rb_mean"] = df_up["5G KPI PCell Layer1 DL RB Num (Including 0)"].mean()
        else:
            feats["rb_mean"] = np.nan
        
        if "Avg MCS" in df_up.columns:
            feats["mcs_mean"] = df_up["Avg MCS"].mean()
        else:
            feats["mcs_mean"] = np.nan
        
    else:
        feats.update({
            "throughput_mean": np.nan,
            "throughput_min": np.nan,
            "throughput_std": np.nan,
            "throughput_p10": np.nan,
            "throughput_low_ratio": np.nan,
            "throughput_drop_ratio": np.nan,
            "sinr_mean": np.nan,
            "sinr_min": np.nan,
            "rsrp_mean": np.nan,
            "rf_good_flag": 0,
            "low_tp_flag": 0,
            "rf_tp_mismatch": 0,
            "pci_switches": 0,
            "rsrp_gap": np.nan,
            "serving_better_ratio": np.nan,
            "cce_fail_mean": np.nan,
            "cce_fail_max": np.nan,
            "cce_fail_high_ratio": np.nan,
            "grant_mean": np.nan,
            "rb_mean": np.nan,
            "mcs_mean": np.nan
        })
    
    # ---------- SIGNALING ----------
    df_sig = parse_csv_block(d.get("signaling_plane_data"))
    
    if df_sig is not None:
        feats["ho_count"] = (df_sig["Event Name"] == "NRHandoverAttempt").sum()
        feats["a3_count"] = (df_sig["Event Name"] == "NREventA3").sum()
        feats["rrc_reest"] = (df_sig["Event Name"] == "NRRRCReestablishAttempt").sum()
    else:
        feats["ho_count"] = 0
        feats["a3_count"] = 0
        feats["rrc_reest"] = 0
    
    # ---------- CONFIG ----------
    df_cfg = parse_csv_block(d.get("network_configuration_data"))
    
    if df_cfg is not None:
        feats["num_cells"] = len(df_cfg)
        feats["avg_power"] = df_cfg["Transmission Power"].mean()
        feats["avg_tilt"] = df_cfg["Mechanical Downtilt"].mean()
        
        # 🔥 Key config feature
        if "PdcchOccupiedSymbolNum" in df_cfg.columns:
            feats["pdcch_symbols"] = pd.to_numeric(df_cfg["PdcchOccupiedSymbolNum"].str.replace("SYM", ""), errors='coerce').mean()
        else:
            feats["pdcch_symbols"] = np.nan
    else:
        feats["num_cells"] = 0
        feats["avg_power"] = np.nan
        feats["avg_tilt"] = np.nan
        feats["pdcch_symbols"] = np.nan
    
    # ---------- TRAFFIC ----------
    df_tr = parse_csv_block(d.get("traffic_data"))
    
    if df_tr is not None:
        feats["dl_prb_util"] = df_tr["Downlink PRB utilization(%)"].mean()
        feats["ul_prb_util"] = df_tr["Uplink PRB utilization(%)"].mean()
        
        # 🔥 CCE stats (VERY IMPORTANT)
        if "Downlink CCE Allocation Success Rate(%)" in df_tr.columns:
            feats["dl_cce_success"] = df_tr["Downlink CCE Allocation Success Rate(%)"].mean()
        else:
            feats["dl_cce_success"] = np.nan
        
        if "Downlink CCE utilization(%)" in df_tr.columns:
            feats["dl_cce_util"] = df_tr["Downlink CCE utilization(%)"].mean()
        else:
            feats["dl_cce_util"] = np.nan
    else:
        feats["dl_prb_util"] = np.nan
        feats["ul_prb_util"] = np.nan
        feats["dl_cce_success"] = np.nan
        feats["dl_cce_util"] = np.nan
    
    # ---------- MR ----------
    df_mr = parse_csv_block(d.get("mr_data"))
    
    if df_mr is not None:
        feats["mr_throughput_mean"] = df_mr["Throughput(Mbps)"].mean()
        feats["mr_rsrp_mean"] = df_mr["Serving RSRP(dBm)"].mean()
    else:
        feats["mr_throughput_mean"] = np.nan
        feats["mr_rsrp_mean"] = np.nan
    
    # ---------- NEIGHBOR MISSING ----------
    try:
        observed = set()
        for col in df_up.columns:
            if "Neighbor Cell" in col and "PCI" in col:
                observed.update(df_up[col].dropna().astype(str))
        
        config_neighbors = set()
        for x in df_cfg["PCell Neighbor Cell (gNodeBID_ARFCN_PCI)"]:
            if isinstance(x, str):
                parts = x.strip("[]").split(",")
                for p in parts:
                    config_neighbors.add(p.split("_")[-1])
        
        feats["missing_neighbors"] = len(observed - config_neighbors)
    except:
        feats["missing_neighbors"] = 0
    
    # ---------- 🔥 FINAL COMPOSITE FEATURE ----------
    # Strong indicator of PDCCH problem
    try:
        feats["pdcch_problem_score"] = (
            (feats["cce_fail_mean"] if not np.isnan(feats["cce_fail_mean"]) else 0) * 0.5 +
            (1 - feats["dl_cce_success"]/100 if not np.isnan(feats["dl_cce_success"]) else 0) * 0.3 +
            feats["rf_tp_mismatch"] * 0.2
        )
    except:
        feats["pdcch_problem_score"] = np.nan
    
    return feats

In [275]:
X = pd.DataFrame([extract_features(s) for s in data])
X = X.fillna(0)

In [276]:
Y = np.array([build_multilabel(s) for s in data])

In [277]:
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

model = MultiOutputClassifier(RandomForestClassifier(n_estimators=200, random_state=42))
model.fit(X_train, y_train)

/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib 

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,RandomForestC...ndom_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the f

In [278]:
X_train

,throughput_mean,throughput_min,throughput_std,throughput_p10,throughput_low_ratio,throughput_drop_ratio,sinr_mean,sinr_min,rsrp_mean,rf_good_flag,...,avg_tilt,pdcch_symbols,dl_prb_util,ul_prb_util,dl_cce_success,dl_cce_util,mr_throughput_mean,mr_rsrp_mean,missing_neighbors,pdcch_problem_score
968,248.230833,29.19,222.685072,31.121,0.500000,0.050703,8.324167,0.28,-98.190000,0,...,5.500000,1.0,12.351575,14.092975,94.854000,0.121900,68.285714,-103.657143,1,0.049188
240,239.074167,23.85,186.953806,49.907,0.416667,0.044109,8.212500,-3.06,-82.356667,1,...,3.250000,1.0,10.565150,10.435075,95.744250,0.170625,112.187500,-81.637500,1,0.059851
819,254.097500,57.43,206.146244,58.523,0.500000,0.100199,12.442500,7.96,-89.684167,1,...,7.500000,1.0,32.202150,35.443900,95.294500,0.189175,50.800000,-87.971429,1,0.067033
692,173.595000,64.83,178.323293,66.649,0.750000,0.133047,7.060000,2.31,-91.877500,0,...,5.923077,1.0,12.728623,11.036592,94.470615,0.149646,114.060000,-94.900000,1,0.067421
420,257.497500,58.93,203.602603,61.067,0.500000,0.098382,12.725000,8.14,-90.166667,1,...,11.500000,1.0,83.605450,86.779250,95.521500,0.165925,51.700000,-87.771429,1,0.063019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1130,330.902500,32.87,220.260021,39.111,0.333333,0.055446,8.971667,-3.91,-84.695000,1,...,8.250000,1.0,9.231050,12.271500,92.708000,0.229250,151.387500,-90.850000,1,0.071876
1294,173.207500,55.38,176.335966,63.596,0.750000,0.100433,7.521667,2.69,-91.647500,0,...,7.461538,1.0,12.827815,13.706354,92.856846,0.183462,83.428571,-94.714286,1,0.073929
860,213.261667,57.38,144.992209,68.314,0.500000,0.138395,11.695833,7.87,-89.720833,1,...,7.000000,1.0,28.930925,33.647150,93.800750,0.201850,55.442857,-87.857143,1,0.069014
1459,255.360000,49.08,191.626841,62.369,0.500000,0.088067,12.852500,7.29,-89.950000,1,...,1.750000,1.0,9.922600,13.776725,92.685000,0.160675,96.150000,-91.316667,1,0.071112


In [279]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

for i, label in enumerate(LABEL_SPACE):
    print(f"\n=== {label} ===")
    print(classification_report(y_test[:, i], y_pred[:, i]))


=== add_neighbor ===
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       352
         1.0       1.00      1.00      1.00        48

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400


=== decrease_power ===
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       372
         1.0       1.00      1.00      1.00        28

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400


=== increase_power ===
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       368
         1.0       1.00      1.00      1.00        32

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weig

/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib 

In [280]:
exact_match = (y_pred == y_test).all(axis=1).mean()
print("Exact match:", exact_match)

Exact match: 1.0


In [281]:
def iou_score(y_true, y_pred):
    scores = []
    for t, p in zip(y_true, y_pred):
        intersection = np.sum((t == 1) & (p == 1))
        union = np.sum((t == 1) | (p == 1))
        if union == 0:
            scores.append(1)
        else:
            scores.append(intersection / union)
    return np.mean(scores)

print("Mean IoU:", iou_score(y_test, y_pred))

Mean IoU: 1.0


In [282]:
import pandas as pd

for i, label in enumerate(LABEL_SPACE):
    clf = model.estimators_[i]
    
    importances = pd.Series(
        clf.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)
    
    print(f"\n=== {label} ===")
    print(importances.head(10))


=== add_neighbor ===
rrc_reest               0.210425
throughput_low_ratio    0.202655
a3_count                0.148464
throughput_mean         0.125704
ho_count                0.059520
rsrp_gap                0.051423
serving_better_ratio    0.047538
sinr_min                0.046625
rsrp_mean               0.018091
pci_switches            0.016742
dtype: float64

=== decrease_power ===
ho_count                0.130897
sinr_min                0.127243
serving_better_ratio    0.117894
rsrp_mean               0.091898
mcs_mean                0.091043
rsrp_gap                0.083504
throughput_low_ratio    0.071159
mr_throughput_mean      0.056480
a3_count                0.051844
sinr_mean               0.036108
dtype: float64

=== increase_power ===
mr_rsrp_mean            0.266267
rsrp_mean               0.261066
serving_better_ratio    0.084260
throughput_min          0.052162
rsrp_gap                0.049727
sinr_min                0.045589
a3_count                0.040038
throughpu

/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib 

In [283]:
from sklearn.tree import export_text

tree = model.estimators_[0].estimators_[0]  # first tree of first label

print(export_text(tree, feature_names=list(X.columns)))

|--- ho_count <= 0.50
|   |--- throughput_low_ratio <= 0.42
|   |   |--- class: 1.0
|   |--- throughput_low_ratio >  0.42
|   |   |--- class: 0.0
|--- ho_count >  0.50
|   |--- class: 0.0



In [284]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.multioutput import MultiOutputClassifier

model_tree = MultiOutputClassifier(
    DecisionTreeClassifier(max_depth=4)
)

model_tree.fit(X_train, y_train)

y_pred = model_tree.predict(X_test)
print("Tree IoU:", iou_score(y_test, y_pred))

Tree IoU: 0.99625


/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib 

In [285]:
for i, label in enumerate(LABEL_SPACE):
    acc = (y_test[:, i] == y_pred[:, i]).mean()
    print(f"{label}: {acc:.3f}")

add_neighbor: 1.000
decrease_power: 0.998
increase_power: 1.000
decrease_threshold: 1.000
tilt_down: 0.998
modify_pdcch: 1.000
adjust_azimuth: 1.000
increase_a3: 1.000
tilt_up: 0.998
decrease_a3: 1.000
check_transport: 1.000
insufficient_data: 1.000


In [286]:
from sklearn.tree import export_text

for i, label in enumerate(LABEL_SPACE):
    print(f"\n=== {label} ===")
    tree = model_tree.estimators_[i]
    print(export_text(tree, feature_names=list(X.columns), decimals=3))


=== add_neighbor ===
|--- throughput_low_ratio <= 0.375
|   |--- class: 1.0
|--- throughput_low_ratio >  0.375
|   |--- class: 0.0


=== decrease_power ===
|--- ho_count <= 2.000
|   |--- serving_better_ratio <= 0.542
|   |   |--- class: 1.0
|   |--- serving_better_ratio >  0.542
|   |   |--- ho_count <= 0.500
|   |   |   |--- sinr_min <= -0.145
|   |   |   |   |--- class: 0.0
|   |   |   |--- sinr_min >  -0.145
|   |   |   |   |--- class: 1.0
|   |   |--- ho_count >  0.500
|   |   |   |--- class: 0.0
|--- ho_count >  2.000
|   |--- class: 1.0


=== increase_power ===
|--- mr_rsrp_mean <= -97.995
|   |--- class: 1.0
|--- mr_rsrp_mean >  -97.995
|   |--- class: 0.0


=== decrease_threshold ===
|--- num_cells <= 9.000
|   |--- class: 0.0
|--- num_cells >  9.000
|   |--- class: 1.0


=== tilt_down ===
|--- ho_count <= 2.000
|   |--- serving_better_ratio <= 0.542
|   |   |--- class: 1.0
|   |--- serving_better_ratio >  0.542
|   |   |--- ho_count <= 0.500
|   |   |   |--- sinr_min <= -0.1

In [287]:
import numpy as np

def tree_rule_predict(feats):
    pred = {k: 0 for k in LABEL_SPACE}
    
    # ===== add_neighbor =====
    if feats["rrc_reest"] > 0.5:
        pred["add_neighbor"] = 1
    
    # ===== decrease_power =====
    if feats["ho_count"] <= 2:
        if feats["ho_count"] <= 0.5:
            if feats["sinr_min"] > -0.145:
                if feats["rsrp_mean"] > -91.520:
                    pred["decrease_power"] = 1
        # else: class 0
    else:
        pred["decrease_power"] = 1
        pred["tilt_down"] = 1
    
    # ===== increase_power =====
    if feats["rsrp_mean"] <= -94.842:
        pred["increase_power"] = 1
    
    # ===== decrease_threshold =====
    if feats["pci_switches"] > 4.5:
        pred["decrease_threshold"] = 1
    
    # ===== tilt_down =====
    if feats["ho_count"] <= 2:
        if feats["ho_count"] <= 0.5:
            if feats["sinr_min"] > -0.145:
                if feats["rsrp_gap"] <= 3.212:
                    pred["tilt_down"] = 1
        # else: class 0
    else:
        pred["tilt_down"] = 1
    
    # ===== modify_pdcch =====
    if feats["dl_cce_success"] <= 87.746:
        pred["modify_pdcch"] = 1
    
    # ===== adjust_azimuth =====
    if feats["rsrp_mean"] <= -94.842:
        pred["adjust_azimuth"] = 1
    
    # ===== increase_a3 =====
    if feats["ho_count"] > 2:
        pred["increase_a3"] = 1
    
    # ===== tilt_up =====
    if feats["dl_prb_util"] > 22.061:
        if feats["ul_prb_util"] <= 56.989:
            pred["tilt_up"] = 1
    
    # ===== decrease_a3 =====
    if feats["serving_better_ratio"] <= 0.625:
        if feats["sinr_min"] <= -0.740:
            pred["decrease_a3"] = 1
    
    # ===== check_transport =====
    if feats["grant_mean"] <= 1041.250:
        pred["check_transport"] = 1
    
    # ===== insufficient_data =====
    if feats["ul_prb_util"] > 56.989:
        pred["insufficient_data"] = 1
    
    return np.array([pred[k] for k in LABEL_SPACE])

In [288]:
y_tree_rule = np.array([
    tree_rule_predict(f) for _, f in X_train.iterrows()
])

In [289]:
y_tree_rule[0]

array([0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [290]:
y_tree_rule = np.array([
    tree_rule_predict(f) for _, f in X_test.iterrows()
])

print("Tree Rule IoU:", iou_score(y_test, y_tree_rule))

Tree Rule IoU: 1.0


In [291]:
y_tree_rule_all = np.array([
    tree_rule_predict(f) for _, f in X.iterrows()
])

print("Tree Rule IoU:", iou_score(Y, y_tree_rule_all))

Tree Rule IoU: 1.0


In [292]:
for i, label in enumerate(LABEL_SPACE):
    acc = (y_test[:, i] == y_tree_rule[:, i]).mean()
    print(f"{label}: {acc:.3f}")

add_neighbor: 1.000
decrease_power: 1.000
increase_power: 1.000
decrease_threshold: 1.000
tilt_down: 1.000
modify_pdcch: 1.000
adjust_azimuth: 1.000
increase_a3: 1.000
tilt_up: 1.000
decrease_a3: 1.000
check_transport: 1.000
insufficient_data: 1.000


In [293]:
LABEL_SPACE

['add_neighbor',
 'decrease_power',
 'increase_power',
 'decrease_threshold',
 'tilt_down',
 'modify_pdcch',
 'adjust_azimuth',
 'increase_a3',
 'tilt_up',
 'decrease_a3',
 'check_transport',
 'insufficient_data']

In [294]:
def enforce_dependencies(pred):
    idx = {k: i for i, k in enumerate(LABEL_SPACE)}
    
    # power ↔ tilt
    if pred[idx["decrease_power"]] == 1:
        pred[idx["tilt_down"]] = 1
        
    if pred[idx["increase_power"]] == 1:
        pred[idx["adjust_azimuth"]] = 1
    
    return pred

In [295]:
y_tree_rule = np.array([
    enforce_dependencies(tree_rule_predict(f))
    for _, f in X_test.iterrows()
])

print("Tree Rule IoU (fixed):", iou_score(y_test, y_tree_rule))

Tree Rule IoU (fixed): 1.0


In [296]:
###

In [297]:
import re

def extract_option_structure(label):
    label_l = label.lower()
    
    # action type
    if "power" in label_l:
        action = "power"
    elif "tilt" in label_l:
        action = "tilt"
    elif "azimuth" in label_l:
        action = "azimuth"
    elif "a3" in label_l:
        action = "a3_offset"
    elif "pdcch" in label_l:
        action = "pdcch"
    elif "neighbor" in label_l:
        action = "neighbor"
    else:
        action = "other"
    
    # direction
    if "increase" in label_l or "lift" in label_l:
        direction = "increase"
    elif "decrease" in label_l or "press down" in label_l:
        direction = "decrease"
    else:
        direction = "none"
    
    # target cell
    match = re.search(r'(\d+_\d+)', label)
    cell = match.group(1) if match else None
    
    return action, direction, cell

In [298]:
analysis_rows = []

for s in data:
    id_to_label = {opt["id"]: opt["label"] for opt in s["task"]["options"]}
    answers = set(s["answer"].split("|"))
    
    # group options
    groups = {}
    
    for opt in s["task"]["options"]:
        action, direction, cell = extract_option_structure(opt["label"])
        key = (action, direction)
        
        if key not in groups:
            groups[key] = []
        
        groups[key].append({
            "id": opt["id"],
            "cell": cell,
            "is_correct": opt["id"] in answers
        })
    
    # analyze groups
    for (action, direction), opts in groups.items():
        num_same_type = len(opts)
        num_correct = sum(o["is_correct"] for o in opts)
        
        if num_same_type > 1:  # only interesting cases
            analysis_rows.append({
                "scenario_id": s["scenario_id"],
                "action": action,
                "direction": direction,
                "num_options_same_type": num_same_type,
                "num_correct": num_correct
            })

In [299]:
analysis_rows = []

for s in data:
    id_to_label = {opt["id"]: opt["label"] for opt in s["task"]["options"]}
    answers = set(s["answer"].split("|"))
    
    # group options
    groups = {}
    
    for opt in s["task"]["options"]:
        action, direction, cell = extract_option_structure(opt["label"])
        key = (action, direction)
        
        if key not in groups:
            groups[key] = []
        
        groups[key].append({
            "id": opt["id"],
            "cell": cell,
            "is_correct": opt["id"] in answers
        })
    
    # analyze groups
    for (action, direction), opts in groups.items():
        num_same_type = len(opts)
        num_correct = sum(o["is_correct"] for o in opts)
        
        if num_same_type > 1:  # only interesting cases
            analysis_rows.append({
                "scenario_id": s["scenario_id"],
                "action": action,
                "direction": direction,
                "num_options_same_type": num_same_type,
                "num_correct": num_correct
            })

In [300]:
import pandas as pd

df_analysis = pd.DataFrame(analysis_rows)

df_analysis.groupby(["action", "direction"]).agg({
    "num_options_same_type": "mean",
    "num_correct": "mean"
}).sort_values("num_options_same_type", ascending=False)

num_options_same_type  num_correct
action    direction                                    
a3_offset decrease                     2.0       0.1155
          increase                     2.0       0.0710
azimuth   none                         2.0       0.0710
neighbor  none                         2.0       0.1200
other     decrease                     2.0       0.1225
          none                         2.0       0.2455
pdcch     none                         2.0       0.1200
power     decrease                     2.0       0.0785
          increase                     2.0       0.0710
tilt      decrease                     2.0       0.0785
          increase                     2.0       0.1270

In [301]:
df_analysis

,scenario_id,action,direction,num_options_same_type,num_correct
0,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,neighbor,none,2,0
1,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,power,decrease,2,1
2,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,power,increase,2,0
3,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,other,none,2,0
4,08e221e5-3ed8-42ed-b7b3-0fd9dfd8d99e,other,decrease,2,0
...,...,...,...,...,...
21995,6a686272-80c5-4a77-ba51-192998460176,tilt,decrease,2,0
21996,6a686272-80c5-4a77-ba51-192998460176,pdcch,none,2,0
21997,6a686272-80c5-4a77-ba51-192998460176,a3_offset,increase,2,0
21998,6a686272-80c5-4a77-ba51-192998460176,other,decrease,2,0


In [302]:
X_test_test = pd.DataFrame([extract_features(s) for s in data_test])

In [303]:
X_test_test

,throughput_mean,throughput_min,throughput_std,throughput_p10,throughput_low_ratio,throughput_drop_ratio,sinr_mean,sinr_min,rsrp_mean,rf_good_flag,...,avg_tilt,pdcch_symbols,dl_prb_util,ul_prb_util,dl_cce_success,dl_cce_util,mr_throughput_mean,mr_rsrp_mean,missing_neighbors,pdcch_problem_score
0,253.883333,55.33,212.195084,58.097,0.500000,0.096609,8.288333,1.39,-82.304167,1,...,4.600000,1.0,13.360740,15.226740,95.550200,0.211640,46.083333,-88.033333,1,0.056266
1,273.370833,21.98,240.093899,53.743,0.500000,0.037747,8.684167,1.19,-82.791667,1,...,5.200000,1.0,12.055020,10.254840,93.075600,0.195460,45.380000,-81.920000,1,0.074107
2,242.737500,28.34,216.244111,37.529,0.500000,0.047468,9.070000,1.48,-82.536667,1,...,5.600000,1.0,9.900780,12.066560,93.636400,0.215080,54.450000,-80.462500,1,0.071591
3,258.905833,33.13,227.808014,41.874,0.500000,0.057911,9.216667,2.00,-83.150000,1,...,2.400000,1.0,10.259520,9.977740,93.796800,0.123440,40.500000,-89.383333,1,0.057360
4,250.440000,25.41,223.128082,37.324,0.500000,0.042041,8.786667,1.77,-84.368333,1,...,3.800000,1.0,12.226260,8.640200,92.872000,0.199080,43.080000,-87.140000,1,0.073884
5,218.804167,34.99,170.784728,54.086,0.500000,0.081091,7.921667,1.09,-85.029167,0,...,1.500000,1.0,11.153850,10.836450,94.393500,0.158400,70.383333,-94.600000,1,0.079320
6,257.834167,23.37,208.240315,60.787,0.500000,0.043701,7.795000,-1.33,-96.527500,0,...,9.750000,1.0,14.537825,11.371875,93.042500,0.182875,50.533333,-102.833333,1,0.062123
7,240.526667,20.70,209.132995,31.173,0.500000,0.036940,8.176667,-1.54,-97.768333,0,...,4.500000,1.0,12.896925,11.834050,92.924750,0.235875,51.771429,-107.814286,1,0.059142
8,240.335000,20.14,218.676217,31.253,0.500000,0.034956,7.180833,-1.42,-97.175833,0,...,3.750000,1.0,11.266075,11.834250,94.731500,0.145100,56.928571,-108.800000,1,0.059555
9,279.290000,53.97,222.833709,67.704,0.500000,0.092267,8.501667,0.43,-83.690000,1,...,4.000000,1.0,14.601975,13.442325,93.730250,0.166850,53.933333,-88.800000,1,0.076309


In [304]:
throughput_mean: 310.1041666666667
throughput_min: 29.24
throughput_std: 205.501041291692
sinr_mean: 8.983333333333333
sinr_min: -3.89
rsrp_mean: -82.89
pci_switches: 1
rsrp_gap: -0.30666666666666603
ho_count: 0
a3_count: 0 #wrong
rrc_reest: 0 #wrong
num_cells: 4
avg_power: 26.0
avg_tilt: 4.0
dl_prb_util: 12.793975
ul_prb_util: 16.0804
mr_throughput_mean: 143.40000000000003
mr_rsrp_mean: -88.58571428571429
missing_neighbors: 1

In [307]:
data_path_test = '../data/Phase_1/raw/test.json'
with open(data_path_test) as f:
    data_test = json.load(f)

len(data_test)

500

In [308]:
X_hold = pd.DataFrame([extract_features(s) for s in data_test])
X_hold = X_hold.fillna(0)

In [309]:
X_hold

,throughput_mean,throughput_min,throughput_std,throughput_p10,throughput_low_ratio,throughput_drop_ratio,sinr_mean,sinr_min,rsrp_mean,rf_good_flag,...,avg_tilt,pdcch_symbols,dl_prb_util,ul_prb_util,dl_cce_success,dl_cce_util,mr_throughput_mean,mr_rsrp_mean,missing_neighbors,pdcch_problem_score
0,310.104167,29.24,205.501041,33.641,0.333333,0.055984,8.983333,-3.89,-82.890000,1,...,4.000000,1.0,12.793975,16.080400,93.663750,0.171625,143.400000,-88.585714,1,0.072342
1,149.419167,41.70,168.030954,52.611,0.750000,0.073758,6.535000,2.29,-91.900833,0,...,6.230769,1.0,11.897577,13.789885,93.443231,0.188400,91.450000,-94.737500,1,0.076754
2,248.883333,52.94,201.495951,54.688,0.500000,0.095456,11.810833,8.18,-90.007500,1,...,4.000000,1.0,80.306850,88.597000,92.653000,0.157800,52.500000,-86.100000,1,0.076208
3,180.494167,54.23,195.717925,57.692,0.750000,0.100411,5.669167,2.26,-92.257500,0,...,6.615385,1.0,9.971485,14.056377,94.687615,0.172362,129.712500,-94.925000,1,0.054687
4,319.400000,32.04,211.964279,52.808,0.333333,0.053988,9.780000,-2.91,-84.399167,1,...,6.500000,1.0,10.291425,11.016675,93.140500,0.177425,102.000000,-92.060000,1,0.058912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,260.172500,61.09,200.214194,68.151,0.500000,0.101868,13.191667,7.83,-90.578333,1,...,6.500000,1.0,29.019525,33.208075,92.249500,0.137450,61.583333,-91.000000,1,0.066585
496,252.582500,55.87,189.503858,70.589,0.500000,0.111399,12.523333,7.77,-90.152500,1,...,9.500000,1.0,31.643425,26.671325,94.919500,0.186625,47.385714,-87.985714,1,0.058992
497,275.305000,22.65,184.054972,42.002,0.333333,0.042201,9.730833,-3.52,-83.447500,1,...,8.500000,1.0,10.436000,14.421575,92.907500,0.226650,118.942857,-91.857143,1,0.066277
498,301.652500,45.41,196.715871,47.811,0.333333,0.080865,9.689167,-3.76,-82.867500,1,...,4.750000,1.0,9.789875,11.378950,91.136500,0.201675,111.100000,-84.300000,1,0.083674


In [310]:
y_tree_rule_hold = np.array([
    tree_rule_predict(f) for _, f in X_hold.iterrows()
])

In [311]:
y_tree_rule_hold

array([[1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       ...,
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0]], shape=(500, 12))

In [312]:
LABEL_SPACE

['add_neighbor',
 'decrease_power',
 'increase_power',
 'decrease_threshold',
 'tilt_down',
 'modify_pdcch',
 'adjust_azimuth',
 'increase_a3',
 'tilt_up',
 'decrease_a3',
 'check_transport',
 'insufficient_data']

In [313]:
df_hold = pd.DataFrame(y_tree_rule_hold, columns=LABEL_SPACE)

In [314]:
df_hold

,add_neighbor,decrease_power,increase_power,decrease_threshold,tilt_down,modify_pdcch,adjust_azimuth,increase_a3,tilt_up,decrease_a3,check_transport,insufficient_data
0,1,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,1
3,0,0,0,1,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
495,0,0,0,0,0,0,0,0,1,0,0,0
496,0,0,0,0,0,0,0,0,1,0,0,0
497,1,0,0,0,0,0,0,0,0,0,0,0
498,1,0,0,0,0,0,0,0,0,0,0,0


In [315]:
counts = y_tree_rule_hold.sum(axis=0)
summary = dict(zip(LABEL_SPACE, counts))

In [316]:
summary

{'add_neighbor': np.int64(64),
 'decrease_power': np.int64(40),
 'increase_power': np.int64(27),
 'decrease_threshold': np.int64(59),
 'tilt_down': np.int64(40),
 'modify_pdcch': np.int64(64),
 'adjust_azimuth': np.int64(27),
 'increase_a3': np.int64(20),
 'tilt_up': np.int64(50),
 'decrease_a3': np.int64(73),
 'check_transport': np.int64(65),
 'insufficient_data': np.int64(58)}

In [317]:
from scipy.sparse import coo_matrix

In [318]:
co_matrix = y_tree_rule_hold.T @ y_tree_rule_hold

In [319]:
coo_labels = coo_matrix(co_matrix)

In [320]:
label_edges = [
    (LABEL_SPACE[r], LABEL_SPACE[c], v)
    for r, c, v in zip(coo_labels.row, coo_labels.col, coo_labels.data)
    if r != c
]

In [321]:
label_edges

[('decrease_power', 'tilt_down', np.int64(40)),
 ('decrease_power', 'increase_a3', np.int64(20)),
 ('increase_power', 'adjust_azimuth', np.int64(27)),
 ('tilt_down', 'decrease_power', np.int64(40)),
 ('tilt_down', 'increase_a3', np.int64(20)),
 ('adjust_azimuth', 'increase_power', np.int64(27)),
 ('increase_a3', 'decrease_power', np.int64(20)),
 ('increase_a3', 'tilt_down', np.int64(20))]

In [322]:
pd.DataFrame(co_matrix, index=LABEL_SPACE, columns=LABEL_SPACE)

,add_neighbor,decrease_power,increase_power,decrease_threshold,tilt_down,modify_pdcch,adjust_azimuth,increase_a3,tilt_up,decrease_a3,check_transport,insufficient_data
add_neighbor,64,0,0,0,0,0,0,0,0,0,0,0
decrease_power,0,40,0,0,40,0,0,20,0,0,0,0
increase_power,0,0,27,0,0,0,27,0,0,0,0,0
decrease_threshold,0,0,0,59,0,0,0,0,0,0,0,0
tilt_down,0,40,0,0,40,0,0,20,0,0,0,0
modify_pdcch,0,0,0,0,0,64,0,0,0,0,0,0
adjust_azimuth,0,0,27,0,0,0,27,0,0,0,0,0
increase_a3,0,20,0,0,20,0,0,20,0,0,0,0
tilt_up,0,0,0,0,0,0,0,0,50,0,0,0
decrease_a3,0,0,0,0,0,0,0,0,0,73,0,0


In [323]:
# Cell 94 - Paths for 500->550 submission export
import os
import json
import numpy as np
import pandas as pd

RAW_TEST_500_PATH = "/Users/faiqrindha/Documents/ZindiAfrica/Telco-Troubleshooting-Agentic-Challenge/Track A/data/Phase_1/raw/test.json"
SAMPLE_SUBMISSION_PATH = "/Users/faiqrindha/Documents/ZindiAfrica/Telco-Troubleshooting-Agentic-Challenge/Track A/sample_submission.csv"
OUT_SUBMISSION_PATH = "/Users/faiqrindha/Documents/ZindiAfrica/Bismillah/Track A/results/result_TrackA_tuned_550.csv"

with open(RAW_TEST_500_PATH) as f:
    raw_test_500 = json.load(f)

sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("raw_test_500:", len(raw_test_500))
print("sample_submission:", len(sample_submission))
print("df_hold rows:", len(df_hold))



raw_test_500: 500
sample_submission: 550
df_hold rows: 500


In [324]:
# Cell 95 - Decode df_hold (12 labels) into per-scenario C-IDs
from collections import Counter


def category_multiplicity_from_train(train_rows):
    by_cat = {cat: [] for cat in LABEL_SPACE}

    for s in train_rows:
        id_to_label = {o["id"]: o["label"] for o in s["task"]["options"]}
        local = Counter()

        for cid in str(s.get("answer", "")).split("|"):
            if not cid:
                continue
            lbl = id_to_label.get(cid)
            if lbl is None:
                continue
            cat = map_to_category(normalize_label(lbl))
            if cat in LABEL_SPACE:
                local[cat] += 1

        for cat, v in local.items():
            by_cat[cat].append(v)

    out = {}
    for cat, vals in by_cat.items():
        out[cat] = max(1, int(round(float(np.median(vals))))) if vals else 1
    return out


def c_key(cid):
    return int(str(cid).replace("C", ""))


def build_cat_to_ids(scenario):
    out = {cat: [] for cat in LABEL_SPACE}
    for opt in scenario["task"]["options"]:
        cat = map_to_category(normalize_label(opt["label"]))
        if cat in out:
            out[cat].append(opt["id"])
    for cat in out:
        out[cat] = sorted(out[cat], key=c_key)
    return out


def decode_row(row_vals, scenario, cat_mult):
    cat_to_ids = build_cat_to_ids(scenario)

    active = []
    for i, cat in enumerate(LABEL_SPACE):
        if int(row_vals[i]) == 1 and len(cat_to_ids[cat]) > 0:
            active.append(cat)

    if len(active) == 0:
        # fallback: pick first available category in this scenario
        for cat in LABEL_SPACE:
            if len(cat_to_ids[cat]) > 0:
                active = [cat]
                break

    selected = []
    for cat in active:
        ids = cat_to_ids[cat]
        k = min(len(ids), int(cat_mult.get(cat, 1)))
        selected.extend(ids[:k])

    selected = sorted(set(selected), key=c_key)

    tag = scenario.get("tag", "single-answer")
    if tag == "single-answer":
        if len(selected) == 0:
            selected = [scenario["task"]["options"][0]["id"]]
        selected = selected[:1]
    else:
        if len(selected) < 2:
            all_ids = sorted([o["id"] for o in scenario["task"]["options"]], key=c_key)
            for cid in all_ids:
                if cid not in selected:
                    selected.append(cid)
                if len(selected) >= 2:
                    break
        if len(selected) > 4:
            selected = selected[:4]

    selected = sorted(set(selected), key=c_key)
    return "|".join(selected)


if len(df_hold) != len(raw_test_500):
    raise ValueError(f"Row mismatch: df_hold={len(df_hold)} vs raw_test_500={len(raw_test_500)}")

cat_mult = category_multiplicity_from_train(data)

pred_500 = pd.DataFrame({
    "ID": [s["scenario_id"] for s in raw_test_500],
    "Track A": [decode_row(df_hold.iloc[i].values, raw_test_500[i], cat_mult) for i in range(len(raw_test_500))]
})

print("pred_500 rows:", len(pred_500))
print(pred_500.head())



pred_500 rows: 500
                                     ID Track A
0  80e3aa96-815d-4683-980c-16db42eab0ef      C1
1  f55a819f-3fb9-4c8f-8859-a5b1649ff2d5      C4
2  2cd1f674-a411-4860-82c6-a16ba624b172      C8
3  923c459f-2ead-4c18-a204-751574ed6ae3      C1
4  2e83ae4f-dca0-4931-bb13-d13d672e14e1     C21


In [325]:
# Cell 96 - Merge 500 Track A predictions into 550 template and save
sub = sample_submission.copy()
if "scenario_id" in sub.columns and "ID" not in sub.columns:
    sub = sub.rename(columns={"scenario_id": "ID"})

if "Track A" not in sub.columns:
    sub["Track A"] = ""
if "Track B" not in sub.columns:
    sub["Track B"] = ""

pred_map = dict(zip(pred_500["ID"], pred_500["Track A"]))
sub["Track A"] = sub["ID"].map(pred_map).fillna(sub["Track A"]).fillna("")
sub["Track B"] = ""

sub = sub[["ID", "Track A", "Track B"]]
os.makedirs(os.path.dirname(OUT_SUBMISSION_PATH), exist_ok=True)
sub.to_csv(OUT_SUBMISSION_PATH, index=False)

print("saved:", OUT_SUBMISSION_PATH)
print("rows:", len(sub))
print("Track A filled:", (sub["Track A"].astype(str).str.strip() != "").sum())
print("Track B filled:", (sub["Track B"].astype(str).str.strip() != "").sum())
sub.head()



saved: /Users/faiqrindha/Documents/ZindiAfrica/Bismillah/Track A/results/result_TrackA_tuned_550.csv
rows: 550
Track A filled: 500
Track B filled: 0


,ID,Track A,Track B
0,80e3aa96-815d-4683-980c-16db42eab0ef,C1,
1,f55a819f-3fb9-4c8f-8859-a5b1649ff2d5,C4,
2,2cd1f674-a411-4860-82c6-a16ba624b172,C8,
3,923c459f-2ead-4c18-a204-751574ed6ae3,C1,
4,2e83ae4f-dca0-4931-bb13-d13d672e14e1,C21,
